# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [13]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 10000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/02 21:41,0,805F65FC0,1768,80611EFF0,25.490000,Euro,25.490000,Euro,ACH,0
1,2022/09/07 09:09,1439,800A61310,1231,800AB9E70,239.420000,US Dollar,239.420000,US Dollar,Cheque,0
2,2022/09/03 10:35,25400,802E3DEC0,25400,802E3DEC0,70.780000,US Dollar,60.410000,Euro,ACH,0
3,2022/09/08 16:45,170412,819F130A1,273676,81BB355A1,0.330591,Bitcoin,0.330591,Bitcoin,Bitcoin,0
4,2022/09/08 07:01,17353,80CDACFC0,56695,8170F4D40,94.520000,Euro,94.520000,Euro,Cheque,0


In [14]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 5447


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/02 21:41,0,805F65FC0,1768,80611EFF0,25.49,Euro,25.49,Euro,ACH,0
2,2022/09/03 10:35,25400,802E3DEC0,25400,802E3DEC0,70.78,US Dollar,60.41,Euro,ACH,0
5,2022/09/01 02:47,48719,8126F69B0,49700,812880FC0,6050.05,Brazil Real,6050.05,Brazil Real,Credit Card,0
6,2022/09/02 23:36,110593,8044268E0,9976,804F7BD80,384041.31,Yuan,384041.31,Yuan,Cheque,0
8,2022/09/01 07:01,117861,807210440,117861,807210440,2306.72,Euro,2306.72,Euro,Reinvestment,0


In [15]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 781


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/02 21:41,0,805F65FC0,1768,80611EFF0,25.49,Euro,25.49,Euro,ACH,0
2,2022/09/03 10:35,25400,802E3DEC0,25400,802E3DEC0,70.78,US Dollar,60.41,Euro,ACH,0
10,2022/09/02 06:41,31746,8133157B0,130808,813315760,599.09,Euro,599.09,Euro,ACH,0
11,2022/09/01 00:19,425,8030A4030,229637,8149BE530,171503.23,Euro,171503.23,Euro,ACH,0
56,2022/09/01 11:04,135933,80D4CBC60,36242,80D4CCA50,650397.00,Canadian Dollar,650397.00,Canadian Dollar,ACH,0


In [16]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [17]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 16


,Timestamp,Receiving Currency,Amount Received,amount_usd
788,2022/09/01 16:58,Yuan,0.07,0.010146
879,2022/09/01 00:00,US Dollar,0.01,0.010000
2046,2022/09/01 03:30,Euro,0.32,0.320128
2088,2022/09/02 20:32,Rupee,5.61,0.070243
3837,2022/09/05 03:59,US Dollar,0.06,0.060000


In [18]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv("output_q5.csv", index=False)
print("Guardado en output_q5.csv")


Q5 resultado: 16 transacciones
Guardado en output_q5.csv
